In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [4]:
df = pd.read_csv("../data/raw/ml_training_dataset.csv")

In [5]:
df.head()

,Application_ID,Age,Gender,City,Employment_Type,Annual_Income,Employment_Duration_Years,Number_of_Dependents,Loan_Purpose,Loan_Amount,...,Previous_Defaults,Missed_Payments,Maximum_Days_Past_Due,Recent_Credit_Enquiries,Credit_History_Length,Number_of_Credit_Accounts,Payment_History,Credit_Score,Risk_Grade,Decision
0,3000001,21,Female,Jaipur,Business Owner,631600,8.0,3,Home Renovation,263000,...,1,2,23,3,0.5,2,86.16,637,E,Reject
1,3000002,29,Male,Ahmedabad,Business Owner,644100,10.9,3,Medical,651000,...,0,0,0,1,7.7,3,100.00,698,D,Approve
2,3000003,28,Female,Nashik,Salaried,1018600,9.5,1,Personal,913000,...,0,0,0,3,2.3,3,100.00,677,D,Approve
3,3000004,30,Female,Nashik,Contract,901600,8.1,3,Medical,856000,...,0,0,0,0,4.3,4,99.66,699,D,Reject
4,3000005,39,Male,Thane,Salaried,1270100,6.3,0,Personal,463000,...,0,0,0,4,15.5,6,100.00,711,C,Approve


In [6]:
df = df.drop(columns=[
    "Application_ID",
    "City",
    "Gender",
    "Risk_Grade"
])

In [7]:
df.head()

,Age,Employment_Type,Annual_Income,Employment_Duration_Years,Number_of_Dependents,Loan_Purpose,Loan_Amount,Loan_Tenure_Months,Existing_Loans_Count,Total_Outstanding_Debt,...,Credit_Utilization,Previous_Defaults,Missed_Payments,Maximum_Days_Past_Due,Recent_Credit_Enquiries,Credit_History_Length,Number_of_Credit_Accounts,Payment_History,Credit_Score,Decision
0,21,Business Owner,631600,8.0,3,Home Renovation,263000,48,0,0.0,...,23.16,1,2,23,3,0.5,2,86.16,637,Reject
1,29,Business Owner,644100,10.9,3,Medical,651000,48,1,112300.0,...,27.53,0,0,0,1,7.7,3,100.00,698,Approve
2,28,Salaried,1018600,9.5,1,Personal,913000,36,0,0.0,...,31.55,0,0,0,3,2.3,3,100.00,677,Approve
3,30,Contract,901600,8.1,3,Medical,856000,48,2,739800.0,...,18.79,0,0,0,0,4.3,4,99.66,699,Reject
4,39,Salaried,1270100,6.3,0,Personal,463000,36,0,0.0,...,40.82,0,0,0,4,15.5,6,100.00,711,Approve


In [8]:
df.shape

(30000, 23)

In [9]:
X = df.drop(columns=["Decision"])

y = df["Decision"].map({
    "Reject": 0,
    "Approve": 1
})

In [10]:
print(X.shape)
print(y.shape)

(30000, 22)
(30000,)


In [11]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

categorical_cols

C:\Users\svbad\AppData\Local\Temp\ipykernel_12072\1278610254.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


['Employment_Type', 'Loan_Purpose']

In [12]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_cols

['Age',
 'Annual_Income',
 'Employment_Duration_Years',
 'Number_of_Dependents',
 'Loan_Amount',
 'Loan_Tenure_Months',
 'Existing_Loans_Count',
 'Total_Outstanding_Debt',
 'Existing_Monthly_EMI',
 'Debt_to_Income_Ratio',
 'Loan_to_Income_Ratio',
 'Credit_Utilization',
 'Previous_Defaults',
 'Missed_Payments',
 'Maximum_Days_Past_Due',
 'Recent_Credit_Enquiries',
 'Credit_History_Length',
 'Number_of_Credit_Accounts',
 'Payment_History',
 'Credit_Score']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

In [15]:
X_train_cat = encoder.fit_transform(
    X_train[categorical_cols]
)

In [16]:
X_test_cat = encoder.transform(
    X_test[categorical_cols]
)

In [17]:
encoded_columns = encoder.get_feature_names_out(
    categorical_cols
)

encoded_columns

array(['Employment_Type_Business Owner', 'Employment_Type_Contract',
       'Employment_Type_Professional', 'Employment_Type_Salaried',
       'Employment_Type_Self-Employed', 'Loan_Purpose_Business',
       'Loan_Purpose_Consumer Durable', 'Loan_Purpose_Education',
       'Loan_Purpose_Home', 'Loan_Purpose_Home Renovation',
       'Loan_Purpose_Medical', 'Loan_Purpose_Personal',
       'Loan_Purpose_Vehicle'], dtype=object)

In [18]:
X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_columns,
    index=X_train.index
)

In [19]:

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_columns,
    index=X_test.index
)

In [20]:
X_train_final = pd.concat(
    [
        X_train[numerical_cols],
        X_train_cat
    ],
    axis=1
)

In [21]:
X_test_final = pd.concat(
    [
        X_test[numerical_cols],
        X_test_cat
    ],
    axis=1
)

In [22]:
print(
    X_train_final.select_dtypes(
        include=["object"]
    ).columns
)

Index([], dtype='str')


In [23]:
print(X_train_final.shape)
print(X_test_final.shape)

(24000, 33)
(6000, 33)


In [24]:
joblib.dump(
    encoder,
    "../models/encoder.pkl"
)

['../models/encoder.pkl']

In [25]:
X_train_final.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_final.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)